# Synthetic A/B Test: Checkout Page Redesign

The Olist dataset has no real randomized experiment, so this module
simulates one with a **known ground truth** — a fake checkout-page
redesign test — to demonstrate the textbook mechanics cleanly: an
a-priori power analysis, randomized assignment, and a post-hoc
significance test that should recover the injected effect.

**Scenario:** current checkout converts visitors to completed purchases
at 10%. The redesign is hypothesized to lift that to 12%.

In [1]:
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (
    proportions_ztest,
    proportion_confint,
    proportion_effectsize,
)

p_control = 0.10
p_treatment = 0.12
alpha = 0.05
power = 0.80

## Power analysis (before running the test)

How many visitors per group do we need to reliably detect a 10% → 12%
lift, at 95% confidence and 80% power?

In [2]:
effect_size = proportion_effectsize(p_treatment, p_control)
analysis = NormalIndPower()
n_required = analysis.solve_power(
    effect_size=effect_size, alpha=alpha, power=power, ratio=1.0, alternative="two-sided"
)

print(f"Cohen's h effect size: {effect_size:.4f}")
print(f"Required sample size per group: {int(np.ceil(n_required))}")

Cohen's h effect size: 0.0640
Required sample size per group: 3835


## Simulate the experiment

Using a fixed random seed for reproducibility. Assigning 4000 visitors
per group (above the ~3835 required by the power analysis).

In [3]:
np.random.seed(42)
n_per_group = 4000

control_conversions = np.random.binomial(1, p_control, n_per_group)
treatment_conversions = np.random.binomial(1, p_treatment, n_per_group)

control_count = int(control_conversions.sum())
treatment_count = int(treatment_conversions.sum())

print(f"control: {control_count}/{n_per_group} = {control_count / n_per_group:.4f}")
print(f"treatment: {treatment_count}/{n_per_group} = {treatment_count / n_per_group:.4f}")

control: 386/4000 = 0.0965
treatment: 470/4000 = 0.1175


## Significance test and confidence intervals

In [4]:
z_stat, p_value = proportions_ztest(
    [treatment_count, control_count], [n_per_group, n_per_group]
)

ci_treatment = proportion_confint(treatment_count, n_per_group, alpha=0.05, method="wilson")
ci_control = proportion_confint(control_count, n_per_group, alpha=0.05, method="wilson")

print(f"z = {z_stat:.4f}, p = {p_value:.5f}")
print(f"treatment 95% CI: ({ci_treatment[0]:.4f}, {ci_treatment[1]:.4f})")
print(f"control 95% CI:   ({ci_control[0]:.4f}, {ci_control[1]:.4f})")
print()
print("Result: statistically significant at alpha=0.05, correctly recovering")
print("the injected effect — confirming the pre-registered sample size from")
print("the power analysis was sufficient to detect it.")

z = 3.0382, p = 0.00238
treatment 95% CI: (0.1079, 0.1278)
control 95% CI:   (0.0877, 0.1060)

Result: statistically significant at alpha=0.05, correctly recovering
the injected effect — confirming the pre-registered sample size from
the power analysis was sufficient to detect it.
